# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
from typing import List

from dotenv import load_dotenv
import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from pydantic import BaseModel, Field

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import AIMessage, ToolMessage, BaseMessage
from lib.tooling import tool
from lib.state_machine import Run


In [3]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


In [4]:
# === phase1: reconnect to persisted Chroma collection ===
# Reconnect to the persisted collection from notebook 1. Embedding function MUST
# match notebook 1 exactly (same model_name AND api_base) or queries will silently
# return wrong results.
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=os.getenv("OPENAI_BASE_URL") or None,
    model_name="text-embedding-3-small",
)
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)
print(f"Connected to udaplay collection with {collection.count()} documents.")


Connected to udaplay collection with 15 documents.


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
@tool
def retrieve_game(query: str) -> str:
    """Semantic search over the UdaPlay video game knowledge base.

    Source: UdaPlay internal game database (15 curated game records as JSON files).
    Each returned record includes its game ID, Name, Platform, YearOfRelease,
    Publisher, Genre, and Description so the answer can cite specific records.

    args:
      query: a question about the video game industry
    """
    result = collection.query(query_texts=[query], n_results=5)
    records = []
    for meta, dist in zip(result['metadatas'][0], result['distances'][0]):
        records.append({
            "id": meta.get("id"),
            "Name": meta.get("Name"),
            "Platform": meta.get("Platform"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Publisher": meta.get("Publisher"),
            "Genre": meta.get("Genre"),
            "Description": meta.get("Description"),
            "distance": round(float(dist), 4),
        })
    return json.dumps(records)


#### Evaluate Retrieval Tool

In [6]:
class EvaluationReport(BaseModel):
    """Structured judgment of whether retrieved docs can answer the question."""
    useful: bool = Field(description="True if the documents contain enough information to answer the question fully.")
    description: str = Field(description="Detailed explanation of the judgment, including what is missing if not useful.")

@tool
def evaluate_retrieval(question: str, retrieved_docs: List[str]) -> str:
    """Assess whether the retrieved documents are sufficient to answer the question.

    Returns a JSON object with fields:
      - useful: bool
      - description: str (explanation)

    args:
      question: the original user question
      retrieved_docs: the documents returned from retrieve_game (as JSON strings or text snippets)
    """
    judge = LLM(model="gpt-4o-mini", temperature=0.0)
    docs_text = "\n---\n".join(retrieved_docs) if retrieved_docs else "(no documents)"
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{docs_text}\n\n"
        "Be strict: if the documents do not directly contain the information required "
        "to answer the question, mark useful=false."
    )
    ai_msg = judge.invoke(prompt, response_format=EvaluationReport)
    report = EvaluationReport.model_validate_json(ai_msg.content)
    return report.model_dump_json()


#### Game Web Search Tool

In [7]:
@tool
def game_web_search(question: str) -> str:
    """Search the web for game-industry information when the internal DB is insufficient.

    Source: Tavily web search API. Results include URLs that should be cited in the answer.

    args:
      question: a question about the game industry
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query=question, max_results=3)
    results = [
        {
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content"),
        }
        for r in response.get("results", [])
    ]
    return json.dumps(results)


In [8]:
# === phase1: structured query reporter ===
def display_query_report(query: str, run: Run) -> None:
    """Structured 4-section report: Tool Calls, Reasoning Trace, Final Answer, Sources."""
    bar = "=" * 70
    print(bar)
    print(f"QUERY: {query}")
    print(bar)

    # Use Agent.get_new_messages so the report shows only THIS run's tool calls,
    # reasoning, and sources -- not cumulative session history.
    messages = Agent.get_new_messages(run)

    tool_results_by_call_id = {
        m.tool_call_id: m.content for m in messages if isinstance(m, ToolMessage)
    }

    print("\n[1] Tool Calls (in order)")
    tool_call_count = 0
    internal_ids = set()
    web_urls = []
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            for tc in m.tool_calls:
                tool_call_count += 1
                args = json.loads(tc.function.arguments)
                args_preview = ', '.join(f'{k}={v!r}' for k, v in args.items())[:120]
                print(f"  - {tc.function.name}({args_preview})")
                raw_result = tool_results_by_call_id.get(tc.id, "")
                # lib/agents.py double-encodes: tool returns a string, then ToolMessage
                # wraps it with json.dumps. So parse twice to get the structured data.
                try:
                    unwrapped = json.loads(raw_result) if isinstance(raw_result, str) else raw_result
                    parsed = json.loads(unwrapped) if isinstance(unwrapped, str) else unwrapped
                except Exception:
                    parsed = raw_result
                if tc.function.name == "retrieve_game" and isinstance(parsed, list):
                    summary = ", ".join(f"{r.get('id')} {r.get('Name')}" for r in parsed[:3])
                    print(f"      -> Retrieved {len(parsed)} games: {summary}" + ("..." if len(parsed) > 3 else ""))
                    internal_ids.update(r.get("id") for r in parsed if r.get("id"))
                elif tc.function.name == "evaluate_retrieval" and isinstance(parsed, dict):
                    print(f"      -> useful: {parsed.get('useful')}")
                    desc = parsed.get('description', '')
                    print(f"      -> description: {desc[:200]}" + ("..." if len(desc) > 200 else ""))
                elif tc.function.name == "game_web_search" and isinstance(parsed, list):
                    print(f"      -> {len(parsed)} web results:")
                    for r in parsed:
                        print(f"         - {r.get('title')} ({r.get('url')})")
                        if r.get("url"):
                            web_urls.append(r["url"])
                else:
                    print(f"      -> {str(parsed)[:200]}")
    if tool_call_count == 0:
        print("  (none -- agent answered without tools)")

    print("\n[2] Reasoning Trace")
    for m in messages:
        if isinstance(m, AIMessage) and m.content:
            print(f"  -> {m.content}")

    print("\n[3] Final Answer")
    final_answer = None
    for m in reversed(messages):
        if isinstance(m, AIMessage) and m.content:
            final_answer = m.content
            break
    print(f"  {final_answer or '(no answer produced)'}")

    print("\n[4] Sources")
    print(f"  Internal: {sorted(internal_ids) if internal_ids else '(none)'}")
    print(f"  Web:      {web_urls if web_urls else '(none)'}")
    print(bar + "\n")


### Agent

In [9]:
INSTRUCTIONS = (
    "You are UdaPlay, a video game industry research assistant. "
    "For EVERY user question, follow this workflow strictly:\n"
    "1. ALWAYS call `retrieve_game` first to search the internal knowledge base.\n"
    "2. ALWAYS call `evaluate_retrieval` next, passing the original question and the "
    "documents returned by step 1.\n"
    "3. If `evaluate_retrieval` returns `useful: false`, you MUST call `game_web_search` "
    "to find additional information from the web.\n"
    "4. Compose a concise final answer with inline citations: for internal records cite "
    "the game Name, Platform, and YearOfRelease (e.g., \"Super Mario 64 (Nintendo 64, 1996)\"); "
    "for web sources include the source URL in parentheses.\n"
    "5. If neither the internal database nor the web yields a confident answer, say so explicitly."
)

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.2,
)


In [10]:
# Queries are taken verbatim from the rubric example list (including original typos)
# to match the grader's expectations.
example_queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
]

# Shared session_id demonstrates the agent's multi-query session memory (rubric line 27).
SESSION_ID = "udaplay_demo"

for q in example_queries:
    run = agent.invoke(q, session_id=SESSION_ID)
    display_query_report(q, run)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


...

[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUERY: Was Mortal Kombat X realeased for Playstation 5?

[1] Tool Calls (in order)
  - retrieve_game(query='Mortal Kombat X release for Playstation 5')
      -> Retrieved 5 games: 005 Marvel's Spider-Man 2, 004 Marvel's Spider-Man, 003 Gran Turismo 5...
  - evaluate_retrieval(question='Was Mortal Kombat X released for Playstation 5?', retrieved_docs=['{"id": "005", "Name": "Marvel\'s Spider-Man)
      -> useful: False
      -> description: The retrieved document discusses 'Marvel's Spider-Man 2' and provides details about its release on PlayStation 5, but it does not mention 'Mortal Kombat X' at all. Therefore, it does not contain any i...
  - game_web_search(question='Was Mortal Kombat X released for Playstation 5?')
      -> 3 web results:
         - Mortal Kombat X - PlayStation (https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved/)
         - Mortal Kombat X - Wikipedia (https://e

### (Optional) Advanced